# ORF 01_noisemodels - JUMP Cell Painting Dataset Noise Model Creation for MicroSplit

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import os
import pandas as pd
from pathlib import Path
from careamics import CAREamist
from careamics.models.lvae.noise_models import GaussianMixtureNoiseModel, create_histogram
from careamics.lvae_training.dataset import DataSplitType
from careamics.config import GaussianMixtureNMConfig, create_n2v_configuration

/home/diya.srivastava/miniforge3/envs/careamics/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define all available channels in the correct order for ORF dataset
class Channels:
    DNA = "DNA"
    RNA = "RNA" 
    ER = "ER"
    AGP = "AGP"
    Mito = "Mito"

# List of all channels in the fixed order: ["DNA", "RNA", "ER", "AGP", "Mito"]
ALL_CHANNELS = [Channels.DNA, Channels.RNA, Channels.ER, Channels.AGP, Channels.Mito]

print("Experiment 3: ORF Noise Model Creation for MicroSplit")
print("="*50)
print(f"Channels to process: {ALL_CHANNELS}")

# Configuration - Set the path to your dataset (relative to current directory)
DATASET_DIR = "orf_microsplit_dataset_300images"  # Change the dataset directory name here

# Derive noise models directory from dataset directory name
dataset_name = os.path.basename(DATASET_DIR)
NOISE_MODELS_DIR = f"noise_models_{dataset_name}"

print(f"Using dataset: {DATASET_DIR}")
print(f"Noise models will be saved to: {NOISE_MODELS_DIR}")

Experiment 3: ORF Noise Model Creation for MicroSplit
Channels to process: ['DNA', 'RNA', 'ER', 'AGP', 'Mito']
Using dataset: orf_microsplit_dataset_300images
Noise models will be saved to: noise_models_orf_microsplit_dataset_300images


In [3]:
# Verify dataset exists
if not os.path.exists(DATASET_DIR):
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

# Load and verify metadata
metadata_path = os.path.join(DATASET_DIR, "dataset_metadata.csv")
if not os.path.exists(metadata_path):
    raise FileNotFoundError(f"Metadata file not found: {metadata_path}")

metadata = pd.read_csv(metadata_path)
print(f"Loaded metadata: {len(metadata)} images")
print(f"Metadata shape: {metadata.shape}")

# Verify all required directories exist
for channel in ALL_CHANNELS:
    channel_dir = os.path.join(DATASET_DIR, channel)
    if not os.path.exists(channel_dir):
        raise FileNotFoundError(f"Channel directory not found: {channel_dir}")
    
    # Count images in this channel directory
    tif_files = [f for f in os.listdir(channel_dir) if f.endswith('.tif')]
    print(f"Channel {channel}: {len(tif_files)} images found")

print("✓ All dataset verification checks passed!")

Loaded metadata: 300 images
Metadata shape: (300, 25)
Channel DNA: 300 images found
Channel RNA: 300 images found
Channel ER: 300 images found
Channel AGP: 300 images found
Channel Mito: 300 images found
✓ All dataset verification checks passed!


In [4]:
# Load data from your ORF microsplit_dataset created in 00_datasets
def load_data(dataset_dir, channel_names, metadata_df=None):
    """
    Load images for each channel from the prepared ORF dataset
    Uses metadata for consistent ordering and validation
    """
    all_images = []
    
    for channel in channel_names:
        print(f"Loading channel: {channel}")
        
        # Find the channel directory
        channel_dir = os.path.join(dataset_dir, channel)
        if not os.path.exists(channel_dir):
            raise FileNotFoundError(f"Channel directory for '{channel}' not found: {channel_dir}")
        
        # Load all tiff files for this channel, sorted by image_id
        files = sorted([f for f in os.listdir(channel_dir) if f.endswith('.tif')])
        
        if len(files) == 0:
            raise ValueError(f"No TIFF files found in {channel_dir}")
        
        # Load each image for this channel
        channel_images = []
        for f in files:
            img_path = os.path.join(channel_dir, f)
            img = tifffile.imread(img_path)
            channel_images.append(img)
        
        # Stack images for this channel
        if channel_images:
            channel_images = np.stack(channel_images)
            all_images.append(channel_images)
            print(f"  Loaded {len(channel_images)} images of shape {channel_images[0].shape}")
        else:
            raise ValueError(f"No images loaded for channel {channel}")
    
    if not all_images:
        raise ValueError(f"No images loaded for any of the channels: {channel_names}")
    
    # Stack along last axis: (n_images, height, width, n_channels)
    result = np.stack(all_images, axis=-1)
    print(f"Final data shape: {result.shape}")
    return result

print("\n" + "="*70)
print("Starting Noise Model Training for Experiment 3: ORF Dataset")
print("="*70)

# Create noise models directory
os.makedirs(NOISE_MODELS_DIR, exist_ok=True)

# Process each channel individually
for channel_idx, channel in enumerate(ALL_CHANNELS):
    print(f"\n\n{'='*50}")
    print(f"Processing channel: {channel} (index {channel_idx})")
    print(f"{'='*50}")
    
    # Check if this channel exists in the dataset
    channel_dir = os.path.join(DATASET_DIR, channel)
    if not os.path.isdir(channel_dir):
        print(f"Channel directory for '{channel}' not found in {DATASET_DIR}")
        continue
    
    print(f"Using dataset: {DATASET_DIR}")
    
    try:
        # Load data for this specific channel only
        input_data = load_data(DATASET_DIR, [channel], metadata)
        print(f"Input data shape: {input_data.shape}")
        
        # Train N2V model for denoising
        config = create_n2v_configuration(
            experiment_name=f"{dataset_name}_noise_models_n2v_{channel}",
            data_type="array",
            axes="SYXC", 
            n_channels=1, # Just one channel at a time
            patch_size=(64, 64),
            batch_size=64,
            num_epochs=10,
        )
        
        # Train N2V on the data
        careamist = CAREamist(source=config, work_dir=f"{NOISE_MODELS_DIR}_{channel}")
        careamist.train(train_source=input_data, val_minimum_split=5)
        
        # Denoise data with the N2V model  
        prediction = careamist.predict(input_data, tile_size=(256, 256))
        
        # Train the Noise Model for this channel
        print(f"Training noise model for channel {channel}")
        channel_data = input_data[..., 0] # Since we're only loading one channel
        channel_prediction = np.concatenate(prediction)[:, 0] # Get the denoised channel
        
        noise_model_config = GaussianMixtureNMConfig(
            model_type="GaussianMixtureNoiseModel",
            min_signal=channel_data.min(),
            max_signal=channel_data.max(),
            n_coeff=4,
            n_gaussian=6
        )
        
        noise_model = GaussianMixtureNoiseModel(noise_model_config)
        noise_model.fit(signal=channel_data, observation=channel_prediction, n_epochs=100)
        
        # Save to ORF-specific noise models directory
        noise_model.save(path=NOISE_MODELS_DIR, name=f"noise_model_{channel}")
        
        # Show the result
        histogram = create_histogram(
            bins=100,
            min_val=channel_data.min(),
            max_val=channel_data.max(),
            signal=channel_data,
            observation=channel_prediction
        )
        print(f"===================")
        print(f"The trained parameters (noise_model_{channel}) is saved at location: {NOISE_MODELS_DIR}")
        
        # Optional: Display some statistics
        print(f"Channel {channel} statistics:")
        print(f"  Signal range: [{channel_data.min():.2f}, {channel_data.max():.2f}]")
        print(f"  Signal mean: {channel_data.mean():.2f}")
        print(f"  Signal std: {channel_data.std():.2f}")
        print(f"  Prediction mean: {channel_prediction.mean():.2f}")
        print(f"  Prediction std: {channel_prediction.std():.2f}")
        
    except Exception as e:
        print(f"Error processing channel {channel}: {str(e)}")
        continue

print("\n" + "="*70)
print("Noise Model Training Complete!")
print("="*70)


Starting Noise Model Training for Experiment 3: ORF Dataset


Processing channel: DNA (index 0)
Using dataset: orf_microsplit_dataset_300images
Loading channel: DNA
  Loaded 300 images of shape (1080, 1080)
Final data shape: (300, 1080, 1080, 1)
Input data shape: (300, 1080, 1080, 1)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A40-16Q') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
Computed dataset mean: [1981.80875046], std: [3469.92038988]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type | Params | Mode 
---------------------------------------
0 | model | UNet | 509 K  | train
---------------------------------------
509 K     Trainable params
0         Non-trainable params
509 K     Total params
2.037     Total estimated model params size (MB)
39        Modules in train mode
0         Modules in eval mode


Epoch 9: 100%|██████████| 1220/1220 [00:47<00:00, 25.53it/s, train_loss_step=0.00128, val_loss=0.000792, train_loss_epoch=0.00748]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 1220/1220 [00:47<00:00, 25.47it/s, train_loss_step=0.00128, val_loss=0.000792, train_loss_epoch=0.00748]


...

Epoch 9: 100%|██████████| 1220/1220 [00:43<00:00, 28.13it/s, train_loss_step=0.0101, val_loss=0.0029, train_loss_epoch=0.0176]  

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 1220/1220 [00:43<00:00, 28.02it/s, train_loss_step=0.0101, val_loss=0.0029, train_loss_epoch=0.0176]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 7500/7500 [00:29<00:00, 251.05it/s]
Training noise model for channel Mito
[GaussianMixtureNoiseModel] min_sigma: 200.0
0 6.034092903137207

The trained parameters (noise_model_Mito) is saved at location: noise_models_orf_microsplit_dataset_300images
The trained parameters (noise_model_Mito) is saved at location: noise_models_orf_microsplit_dataset_300images
Channel Mito statistics:
  Signal range: [300.00, 65535.00]
  Signal mean: 2500.05
  Signal std: 1837.91
  Prediction mean: 2495.29
  Prediction std: 1772.99

Noise Model Training Complete!


In [5]:
# Verify all noise models were created
print(f"\nVerifying noise models in {NOISE_MODELS_DIR}:")
for channel in ALL_CHANNELS:
    noise_model_path = os.path.join(NOISE_MODELS_DIR, f"noise_model_{channel}.npz")
    if os.path.exists(noise_model_path):
        print(f"✓ {channel}: noise_model_{channel}.npz created successfully")
    else:
        print(f"✗ {channel}: noise_model_{channel}.npz NOT FOUND")

print(f"\nNoise models ready for the next step!")
print(f"You can now proceed to the 02_train notebook using:")
print(f"  - Dataset: {DATASET_DIR}")
print(f"  - Noise models: {NOISE_MODELS_DIR}")
print(f"  - Channels: {ALL_CHANNELS}")

# Summary statistics
print(f"\nDataset Summary:")
print(f"  Total images processed: {len(metadata)} per channel")
print(f"  Total channels: {len(ALL_CHANNELS)}")
print(f"  Image dimensions: 1080 × 1280 pixels")
print(f"  Channel combination order: {' → '.join(ALL_CHANNELS)}")


Verifying noise models in noise_models_orf_microsplit_dataset_300images:
✓ DNA: noise_model_DNA.npz created successfully
✓ RNA: noise_model_RNA.npz created successfully
✓ ER: noise_model_ER.npz created successfully
✓ AGP: noise_model_AGP.npz created successfully
✓ Mito: noise_model_Mito.npz created successfully

Noise models ready for the next step!
You can now proceed to the 02_train notebook using:
  - Dataset: orf_microsplit_dataset_300images
  - Noise models: noise_models_orf_microsplit_dataset_300images
  - Channels: ['DNA', 'RNA', 'ER', 'AGP', 'Mito']

Dataset Summary:
  Total images processed: 300 per channel
  Total channels: 5
  Image dimensions: 1080 × 1280 pixels
  Channel combination order: DNA → RNA → ER → AGP → Mito
